# Data Transformations — complex_report

This notebook materialises calculated columns (row-level
Tableau formulas) as physical columns in the Lakehouse
Delta table, so the Semantic Model can reference them
directly via DirectLake.

**Generated:** 2026-03-05 10:46


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime

print(f"Transformations started: {datetime.now()}")


## Calculated Columns (materialised)

5 calculated column(s) will be added
to the `targets` Delta table.


In [ ]:
# Read the main table from Lakehouse
df = spark.read.format("delta").table("targets")
print(f"Rows: {df.count()}, Columns: {len(df.columns)}")


### `Profit` (real)

**Tableau formula:** `[Revenue] - [Cost]`

In [ ]:
df = df.withColumn("Profit", F.col("Revenue") - F.col("Cost"))


### `Priority Label` (string)

**Tableau formula:** `CASE [Priority] WHEN "Critical" THEN "🔴 Critical" WHEN "High" THEN "🟠 High" WHEN "Medium" THEN "🟡 Medium" WHEN "Low" THEN "🟢 Low" ELSE "⚪ Unknown" END`

In [ ]:
df = df.withColumn("Priority Label", CASE F.col("Priority") WHEN "Critical" THEN "🔴 Critical" WHEN "High" THEN "🟠 High" WHEN "Medium" THEN "🟡 Medium" WHEN "Low" THEN "🟢 Low" ELSE "⚪ Unknown" END)


### `Revenue Tier` (string)

**Tableau formula:** `IF [Revenue] > 10000 THEN "Platinum" ELSEIF [Revenue] > 5000 THEN "Gold" ELSEIF [Revenue] > 1000 THEN "Silver" ELSE "Bronze" END`

In [ ]:
df = df.withColumn("Revenue Tier", F.when(F.col("Revenue") > 10000, "Platinum" ELSEIF F.col("Revenue") > 5000 THEN "Gold" ELSEIF F.col("Revenue") > 1000 THEN "Silver").otherwise("Bronze"))


### `Days to Ship` (integer)

**Tableau formula:** `DATEDIFF('day', [OrderDate], [ShipDate])`

In [ ]:
df = df.withColumn("Days to Ship", DATEDIFF('day', F.col("OrderDate"), F.col("ShipDate")))


### `Has Discount` (string)

**Tableau formula:** `IF ZN([Discount]) > 0 THEN "Yes" ELSE "No" END`

In [ ]:
df = df.withColumn("Has Discount", F.when(ZN(F.col("Discount")) > 0, "Yes").otherwise("No"))


In [ ]:
# Overwrite the Delta table with the new columns
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("targets")
print(f"✓ Written {len(df.columns)} columns to targets")


## Measures (DAX — informational only)

The following calculations use aggregation functions
and will remain as DAX measures in the Semantic Model.


- **`Profit Margin`**: `SUM([Revenue] - [Cost]) / SUM([Revenue])`

- **`Net Revenue`**: `SUM(IF [OrderStatus] != "Cancelled" THEN [Revenue] * (1 - [Discount]) ELSE 0 END)`

- **`Customer LTV`**: `{FIXED [CustomerID] : SUM([Revenue])}`

- **`Avg Sales excl Region`**: `{EXCLUDE [RegionName] : AVG([Revenue])}`

- **`Running Revenue`**: `RUNNING_SUM(SUM([Revenue]))`

- **`Budget Variance`**: `(SUM([Revenue]) - SUM([TargetRevenue])) / SUM([TargetRevenue])`

## Summary

In [ ]:
# ── Summary ────────────────────────────────────────────
print("\n" + "=" * 60)
print("Transformations Complete")
print("=" * 60)
print("Calculated columns materialised: 5")
print("Measures kept as DAX:            6")
print(f"Completed: {datetime.now()}")
